In [3]:
import pandas as pd

# Leer el archivo parquet
df = pd.read_parquet('data/comercio_mexico.parquet')

# Ver las primeras 5 filas para confirmar
print(df.head())

  STATE COMMODITY                                               DESC  \
0     -    020130  MEAT OF BOVINE ANIMALS, BONELESS, FRESH OR CHI...   
1    AZ    020130  MEAT OF BOVINE ANIMALS, BONELESS, FRESH OR CHI...   
2    CA    020130  MEAT OF BOVINE ANIMALS, BONELESS, FRESH OR CHI...   
3    FL    020130  MEAT OF BOVINE ANIMALS, BONELESS, FRESH OR CHI...   
4    GA    020130  MEAT OF BOVINE ANIMALS, BONELESS, FRESH OR CHI...   

      VALOR month     flow  year  
0  93548947    01  imports  2025  
1   3445240    01  imports  2025  
2  53176050    01  imports  2025  
3   1743242    01  imports  2025  
4   3269897    01  imports  2025  


In [8]:
import pandas as pd

# Leer el archivo parquet
df = pd.read_parquet('data/ms_total.parquet')

# Ver las primeras 5 filas para confirmar
print(df.head())

  STATE COMMODITY            DESC     VALOR month     flow  year
0     -    010611  PRIMATES, LIVE  15911200    01  imports  2025
1    FL    010611  PRIMATES, LIVE     72000    01  imports  2025
2    MD    010611  PRIMATES, LIVE   7500000    01  imports  2025
3    PA    010611  PRIMATES, LIVE    419200    01  imports  2025
4    TX    010611  PRIMATES, LIVE   7920000    01  imports  2025


In [9]:
import pandas as pd

# Leer el archivo parquet
df = pd.read_parquet('data/xs_total.parquet')

# Ver las primeras 5 filas para confirmar
print(df.head())

  STATE COMMODITY                                    DESC      VALOR month  \
0     -    020319  MEAT OF SWINE, NESOI, FRESH OR CHILLED  114643450    01   
1    AL    020319  MEAT OF SWINE, NESOI, FRESH OR CHILLED      10000    01   
2    AR    020319  MEAT OF SWINE, NESOI, FRESH OR CHILLED    1070712    01   
3    AZ    020319  MEAT OF SWINE, NESOI, FRESH OR CHILLED     101330    01   
4    CA    020319  MEAT OF SWINE, NESOI, FRESH OR CHILLED   13799306    01   

      flow  year  
0  exports  2025  
1  exports  2025  
2  exports  2025  
3  exports  2025  
4  exports  2025  


In [5]:
import pandas as pd
import numpy as np

# 1. Leer el archivo original
df = pd.read_parquet('data/comercio_mexico.parquet')

# 2. LIMPIEZA: Asegurarnos de que 'VALOR' sea un número
# Si la columna es texto, le quitamos las comas y la convertimos a numérico
if df['VALOR'].dtype == 'object':
    df['VALOR'] = df['VALOR'].astype(str).str.replace(',', '', regex=False)
    # errors='coerce' convierte cualquier texto que no se pueda leer en un valor nulo (NaN)
    df['VALOR'] = pd.to_numeric(df['VALOR'], errors='coerce') 

# 3. Generar el multiplicador y realizar la operación
multiplicador = np.random.uniform(1, 10, size=len(df))
df['VALOR'] = df['VALOR'] * multiplicador

# 4. Guardar el resultado en un nuevo archivo parquet
df.to_parquet('data/comercio_total.parquet')

print("Archivo 'comercio_total.parquet' creado exitosamente y valores convertidos.")

Archivo 'comercio_total.parquet' creado exitosamente y valores convertidos.


In [7]:
# Filtrar por el estado de Alaska (AK) y asegurarnos de que el flujo sea 'imports'
df_alaska = df[(df['STATE'] == 'AK') & (df['flow'] == 'imports')]

# Ver los primeros resultados
df_alaska

,STATE,COMMODITY,DESC,VALOR,month,flow,year
6141,AK,730890,"STRUCTURES AND PARTS OF STRUCTURES NESOI, OF I...",4.812652e+04,01,imports,2025
6755,AK,730429,CASING AND TUBING OF A KIND USED IN DRILLING F...,1.261239e+06,01,imports,2025
9483,AK,847150,DIGITAL PROCESSING UNITS OTHER THAN THOSE OF 8...,2.200634e+04,01,imports,2025
10218,AK,854129,"TRANSISTORS, OTHER THAN PHOTOSENSITIVE, NESOI",8.322000e+03,01,imports,2025
12263,AK,841459,"FANS, NESOI",4.933316e+03,01,imports,2025
...,...,...,...,...,...,...,...
1284208,AK,847989,MACHINES AND MECHANICAL APPLIANCES HAVING INDI...,0.000000e+00,05,imports,2026
1285272,AK,902480,MACHINES AND APPLIANCES NESOI FOR TESTING THE ...,0.000000e+00,05,imports,2026
1285812,AK,842132,CATALYTIC CONVERTERS OR PARTICULATE FILTERS FO...,0.000000e+00,05,imports,2026
1285933,AK,821192,"KNIVES, OTHER THAN TABLE KNIVES, HAVING FIXED ...",0.000000e+00,05,imports,2026


In [1]:
import pandas as pd
import os

# 1. Rutas a los archivos (asumiendo que están en la carpeta 'data')
ruta_ms = os.path.join("data", "ms_total.parquet")
ruta_xs = os.path.join("data", "xs_total.parquet")

print("Cargando archivos individuales...")
df_ms = pd.read_parquet(ruta_ms)
df_xs = pd.read_parquet(ruta_xs)

print("Concatenando bases de datos...")
df_tot = pd.concat([df_ms, df_xs], ignore_index=True)

# 2. OPTIMIZACIÓN DE MEMORIA (Convierte textos repetitivos a categorías para bajar el peso del archivo)
columnas_categoricas = ['STATE', 'flow', 'month']
for col in columnas_categoricas:
    if col in df_tot.columns:
        df_tot[col] = df_tot[col].astype('category')

# 3. Guardar el archivo unificado
ruta_salida = os.path.join("data", "comercio_total.parquet")
print(f"Guardando archivo unificado optimizado en {ruta_salida}...")
df_tot.to_parquet(ruta_salida, index=False)

print("¡Proceso completado con éxito!")

Cargando archivos individuales...
Concatenando bases de datos...
Guardando archivo unificado optimizado en data\comercio_total.parquet...
¡Proceso completado con éxito!


In [2]:
import pandas as pd
import os

def limpiar_y_optimizar(df):
    # 1. Limpieza a numérico
    df['VALOR'] = pd.to_numeric(df['VALOR'].astype(str).str.replace(',', '').str.replace(' ', ''), errors='coerce').fillna(0)
    
    # 2. Formateo de fechas
    df['year'] = pd.to_numeric(df['year'], errors='coerce').fillna(0).astype(int)
    df['month'] = df['month'].astype(str).str.zfill(2)
    
    # 3. Crear capítulo
    df['Chapter'] = df['COMMODITY'].astype(str).str.zfill(6).str[:2]
    
    # 4. OPTIMIZACIÓN EXTREMA DE RAM: Convertir texto repetitivo a 'Category'
    columnas_categoricas = ['STATE', 'flow', 'month', 'Chapter']
    for col in columnas_categoricas:
        if col in df.columns:
            df[col] = df[col].astype('category')
            
    return df

print("Procesando datos de México...")
df_mex = pd.read_parquet(os.path.join("data", "comercio_mexico.parquet"))
df_mex_clean = limpiar_y_optimizar(df_mex)

print("Procesando datos Totales (MS y XS)...")
df_ms = pd.read_parquet(os.path.join("data", "ms_total.parquet"))
df_xs = pd.read_parquet(os.path.join("data", "xs_total.parquet"))
df_tot = pd.concat([df_ms, df_xs], ignore_index=True)
df_tot_clean = limpiar_y_optimizar(df_tot)

# Sobrescribimos los archivos originales con las versiones hiper-ligeras
df_mex_clean.to_parquet(os.path.join("data", "comercio_mexico.parquet"), index=False)
df_tot_clean.to_parquet(os.path.join("data", "comercio_total.parquet"), index=False)

print("¡Archivos listos para subir a GitHub!")

Procesando datos de México...
Procesando datos Totales (MS y XS)...
¡Archivos listos para subir a GitHub!
